# Public-source note

Final plotting notebook for the reordered 23-state ATAC ChromHMM model (Fig. S5a). Outputs were stripped and paths sanitized.

In [ ]:
import os, shlex, subprocess, sys
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


In [ ]:

reordered_dir = "/path/to/CEBPB_seq/ChromHMM/All_cellTypes_atac/numstates_23_reordered_V2/"
emissions_fpath = Path(reordered_dir) / "emissions_23_with_Genome_percentage.txt"
seg_fpath_out= Path(reordered_dir) / "ATAC_23_segments.bed"

emissions_out_pdf= Path(reordered_dir) / "ATAC_STACKED_ChromHMM_23_reordered_V2.pdf"

In [ ]:
reordered_dir

In [ ]:
col_order=['Genome %',
'MEF', 'MEF_d2_OSKM','MEF_d2_OSKM_mCEBPA', 'MEF_d2_OSKM_mCEBPB',  'MEF_d2_OSKM_hCEBPD','MEF_d2_mCEBPA', 'MEF_d2_mCEBPB', 
'NSC', 'NSC_d2_OSKM', 'NSC_d2_OSKM_mCEBPA', 'NSC_d2_OSKM_mCEBPB', 'NSC_d2_mCEBPA', 'NSC_d2_mCEBPB',
'ESC', 'ESC_d1_hCEBPA', 'ESC_d2_hCEBPA',   
'Bcell_Graf', 'Bcell_Graf_Cebpa18h', 'Bcell_Graf_OKSMd1', 'Bcell_Graf_OKSMd2'  ]

In [ ]:
import pandas as pd

# load
emissions_df = pd.read_csv(emissions_fpath, sep="\t", index_col=0)
# state_col = "State (Emission order)"

# reorder columns
emissions_df = emissions_df[col_order]

em_mat = emissions_df.copy()
em_mat

In [ ]:
# em_mat = em_mat * 100

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# from matplotlib.colors import LinearSegmentedColormap
# import matplotlib as mpl

# # --- Global style (safe for publications)
# import matplotlib as mpl
# mpl.rcParams.update({
#     "pdf.fonttype": 42,
#     "ps.fonttype": 42,
#     "pdf.use14corefonts": True,   # use standard PDF fonts (Helvetica etc.)
#     "font.family": "sans-serif",
#     "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
# })

# # Ensure values are percent (ChromHMM style)
# # If your values are 0–1:
# em_plot = em_mat.copy() * 100   # remove *100 if already 0–100

# # ChromHMM white → blue colormap
# chromhmm_cmap = LinearSegmentedColormap.from_list(
#     "chromhmm_white_blue",
#     ["#FFFFFF", "#DEEBF7", "#9ECAE1", "#4292C6", "#08519C"]
# )


# fig, ax = plt.subplots(
#     figsize=(0.3 * em_plot.shape[1] + 1.6,
#              0.3 * em_plot.shape[0] + 1.6)
# )


# ax = sns.heatmap(
#     em_plot,
#     cmap=chromhmm_cmap,
#     # cmap="Blues",
#     vmin=0,
#     vmax=100,
#     annot=True,
#     fmt=".0f",
#     linewidths=0.0,
#     linecolor="white",
#     cbar=False
# )

# # Make all annotation text white
# for text in ax.texts:
#     text.set_color("white")
    
        
# # Put marks on top (like ChromHMM)
# ax.xaxis.tick_top()
# ax.xaxis.set_label_position('top')

# ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
# ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

# ax.set_xlabel("")
# ax.set_ylabel("")
# ax.set_title("ChromHMM Emissions", pad=20)

# plt.tight_layout()
# fig.savefig("ATAC_chromhmm_emissions.pdf", bbox_inches="tight")   # vector
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl

# --- Global style
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "pdf.use14corefonts": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
})

# Do NOT multiply by 100
em_plot = em_mat.copy()
# em_plot = em_plot * 100
em_plot = em_mat.copy()
em_plot.iloc[:, 1:] = em_plot.iloc[:, 1:] * 100

# ChromHMM white → blue colormap
chromhmm_cmap = LinearSegmentedColormap.from_list(
    "chromhmm_white_blue",
    ["#FFFFFF", "#DEEBF7", "#9ECAE1", "#4292C6", "#08519C"]
)

fig, ax = plt.subplots(
    figsize=(0.3 * em_plot.shape[1] + 1.6,
             0.3 * em_plot.shape[0] + 1.6)
)

ax = sns.heatmap(
    em_plot,
    cmap=chromhmm_cmap,
    vmin=0,
    vmax=100,
    annot=True,
    fmt=".0f",
    # fmt=".2f",
    linewidths=0.0,
    linecolor="white",
    cbar=False
)

# Force first column cells to be white
for row in range(em_plot.shape[0]):
    ax.add_patch(
        plt.Rectangle(
            (0, row),      # x, y position
            1,             # width
            1,             # height
            fill=True,
            facecolor="white",
            edgecolor="white",
            linewidth=0
        )
    )

# Annotation text colors
for text in ax.texts:
    x, y = text.get_position()

    # First column annotation should be visible on white
    if int(x - 0.5) == 0:
        text.set_color("black")
    else:
        text.set_color("white")

# Put marks on top
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")

ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("ChromHMM Emissions", pad=20)

plt.tight_layout()
# fig.savefig("ATAC_chromhmm_emissions.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
import matplotlib as mpl

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "pdf.use14corefonts": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
})

# Copy data and multiply only the second column onward by 100
em_plot = em_mat.copy()
em_plot.iloc[:, 1:] *= 100

# Separate first column and remaining columns
first_col = em_plot.iloc[:, :1]
remaining_cols = em_plot.iloc[:, 1:]

# Colormaps
white_cmap = ListedColormap(["white"])

chromhmm_cmap = LinearSegmentedColormap.from_list(
    "chromhmm_white_blue",
    ["#FFFFFF", "#DEEBF7", "#9ECAE1", "#4292C6", "#08519C"]
)

# Set subplot widths based on the number of columns
fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    sharey=True,
    gridspec_kw={
        "width_ratios": [1, remaining_cols.shape[1]],
        "wspace": 0.02
    },
    figsize=(
        # 0.3 * em_plot.shape[1] + 1.6,
        0.4 * em_plot.shape[1] + 1.8,
        0.3 * em_plot.shape[0] + 1.6
    )
)

# First-column heatmap
sns.heatmap(
    first_col,
    ax=ax1,
    cmap=white_cmap,
    annot=True,
    fmt=".2f",
    cbar=False,
    linewidths=0,
    linecolor="white",
    annot_kws={"color": "black"}
)

# Remaining-columns heatmap
sns.heatmap(
    remaining_cols,
    ax=ax2,
    cmap=chromhmm_cmap,
    vmin=0,
    vmax=100,
    annot=True,
    fmt=".0f",
    cbar=False,
    linewidths=0,
    linecolor="white",
    annot_kws={"color": "white"}
)

# Put column labels on top
for ax in (ax1, ax2):
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_xlabel("")
    ax.set_ylabel("")

# Display row labels only on the first subplot
ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0)
ax2.tick_params(axis="y", left=False, labelleft=False)

# Remove the visible gap between subplot borders
ax1.spines["right"].set_visible(False)
ax2.spines["left"].set_visible(False)

# fig.suptitle("ChromHMM Emissions", y=1.02)

plt.tight_layout()
fig.savefig("ATAC_chromhmm_emissions.pdf", bbox_inches="tight")
plt.show()